In [ ]:
import cv2                      # Import OpenCV for image processing
import numpy as np              # Import NumPy for mathematical calculations
from google.colab.patches import cv2_imshow   # Used to display images in Google Colab


# -------------------- Eye Landmark Points --------------------

# These are the landmark indexes for the LEFT eye in MediaPipe Face Mesh
LEFT_EYE = [362, 385, 387, 263, 373, 380]

# These are the landmark indexes for the RIGHT eye in MediaPipe Face Mesh
RIGHT_EYE = [33, 160, 158, 133, 153, 144]


# -------------------- Function to Calculate Eye Aspect Ratio (EAR) --------------------

def eye_aspect_ratio(landmarks, eye_points, w, h):
    # landmarks  -> All facial landmarks detected by MediaPipe
    # eye_points -> Landmark indexes of one eye
    # w          -> Width of the image
    # h          -> Height of the image

    points = []      # Empty list to store eye landmark coordinates

    # Loop through each landmark index of the eye
    for index in eye_points:

        # Convert normalized X coordinate (0 to 1) into actual pixel value
        x = int(landmarks[index].x * w)

        # Convert normalized Y coordinate (0 to 1) into actual pixel value
        y = int(landmarks[index].y * h)

        # Store the pixel coordinate of the landmark
        points.append([x, y])


    # -------------------- Calculate Vertical Distances --------------------

    # Distance between landmark 2 and landmark 6
    # Measures one vertical opening of the eye
    A = np.linalg.norm(
        np.array(points[1]) - np.array(points[5])
    )

    # Distance between landmark 3 and landmark 5
    # Measures another vertical opening of the eye
    B = np.linalg.norm(
        np.array(points[2]) - np.array(points[4])
    )


    # -------------------- Calculate Horizontal Distance --------------------

    # Distance between the two eye corners
    # Measures the width of the eye
    C = np.linalg.norm(
        np.array(points[0]) - np.array(points[3])
    )


    # -------------------- Calculate Eye Aspect Ratio --------------------

    # EAR Formula
    # (Vertical Distance 1 + Vertical Distance 2)
    # -------------------------------------------
    #        2 × Horizontal Distance
    ear = (A + B) / (2.0 * C)


    # Return the calculated EAR value
    return ear

In [ ]:
# -------------------- Capture Image --------------------

# Open the webcam and capture a photo
# The image is saved, and its file path is returned.
image_path = take_photo()


# -------------------- Read the Image --------------------

# Load the captured image from the file path using OpenCV
image = cv2.imread(image_path)


# -------------------- Convert BGR to RGB --------------------

# OpenCV loads images in BGR format.
# MediaPipe works with RGB images.
# So we convert the image from BGR to RGB.
rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)


# -------------------- Create MediaPipe Image --------------------

# Convert the NumPy image into a MediaPipe Image object.
# This is the format required by the MediaPipe Face Landmarker.
mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,   # Specify that the image is in RGB format
    data=rgb                            # Pass the RGB image
)


# -------------------- Detect Face Landmarks --------------------

# Run the Face Landmarker model on the image.
# The detector finds all facial landmarks and stores them in 'results'.
results = detector.detect(mp_image)

In [ ]:
# -------------------- Check if a Face is Detected --------------------

# MediaPipe stores all detected faces in results.face_landmarks.
# If at least one face is detected, this condition becomes True.
if results.face_landmarks:

    # Get the landmarks of the first detected face.
    # If multiple faces are detected, index 0 refers to the first face.
    landmarks = results.face_landmarks[0]


    # Get the image dimensions.
    # h = Height of image
    # w = Width of image
    # _ = Number of color channels (ignored here)
    h, w, _ = image.shape


    # -------------------- Calculate Left Eye EAR --------------------

    # Calculate the Eye Aspect Ratio (EAR) for the left eye.
    left_ear = eye_aspect_ratio(
        landmarks,
        LEFT_EYE,
        w,
        h
    )


    # -------------------- Calculate Right Eye EAR --------------------

    # Calculate the Eye Aspect Ratio (EAR) for the right eye.
    right_ear = eye_aspect_ratio(
        landmarks,
        RIGHT_EYE,
        w,
        h
    )


    # -------------------- Average EAR --------------------

    # Calculate the average EAR of both eyes.
    # This gives a more stable result.
    avg_ear = (left_ear + right_ear) / 2


    # Print the EAR value up to 3 decimal places.
    print("EAR :", round(avg_ear, 3))


    # -------------------- Drowsiness Threshold --------------------

    # If EAR is less than 0.20,
    # it means the eyes are almost closed.
    if avg_ear < 0.20:

        # Display DROWSINESS DETECTED in RED.
        cv2.putText(
            image,
            "DROWSINESS DETECTED",
            (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

    else:

        # Otherwise display NORMAL in GREEN.
        cv2.putText(
            image,
            "NORMAL",
            (50, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )


    # -------------------- Draw Face Landmarks --------------------

    # Loop through all 468 facial landmarks.
    for landmark in landmarks:

        # Convert normalized coordinates into pixel coordinates.
        x = int(landmark.x * w)
        y = int(landmark.y * h)

        # Draw a small blue circle on each landmark.
        cv2.circle(
            image,
            (x, y),
            1,
            (255, 0, 0),
            -1
        )


# -------------------- If No Face is Detected --------------------

else:

    # Display NO FACE DETECTED in RED.
    cv2.putText(
        image,
        "NO FACE DETECTED",
        (50, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2)


# -------------------- Display Final Image --------------------

# Show the output image in Google Colab.
cv2_imshow(image)